## Importar librerías necesarias

In [1]:
import transformers
print(transformers.__version__)

4.57.0


In [2]:
#!pip install transformers
#!pip install timm      #requerido en facebook/dinov2-base
#!pip install --upgrade transformers
#!pip install scikit-learn
#!pip install accelerate>=0.26.0

In [3]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
from transformers import AutoFeatureExtractor, AutoModelForImageClassification
from PIL import Image, ImageFile
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss
from tqdm import tqdm
import os
from datasets import Dataset, load_from_disk
from transformers import Trainer
from transformers import TrainingArguments


import numpy as np
import sklearn.metrics

## Usar GPU

In [4]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Usando dispositivo:", DEVICE)

Usando dispositivo: cpu


## Cargar modelo preentrenado

In [5]:
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

In [6]:
model_names = [
    "microsoft/swinv2-base-patch4-window16-256",
    "microsoft/beit-base-patch16-224",
    "google/vit-base-patch16-224",
    "facebook/convnext-base", #224
    "facebook/dinov2-base" #224
]

# Tamaño de imagen para swinv2
image_size=256

# Cambia este índice para elegir el modelo
model_index = 0
model_name = model_names[model_index]

# Nombre para guardar pesos / logs
model_output_name = model_name.split("/")[-1]
model_output = f"./models/{model_output_name}"

# Cargar modelo
try:
    model = AutoModelForImageClassification.from_pretrained(model_output)
    print(f"Modelo cargado desde {model_output}")
except:
    model = AutoModelForImageClassification.from_pretrained(model_name)
    model.config.num_labels = model.num_labels = 4
    print(f"Modelo {model_name} descargado de Hugging Face")
    if "swinv2" in model_name:
        model.classifier = nn.Linear(model.swinv2.num_features, model.num_labels)


Modelo microsoft/swinv2-base-patch4-window16-256 descargado de Hugging Face


## Preparacion de datos (aumentaciones básicas)

In [17]:
dataset = load_from_disk("../dataset") #dataset se sube aparte
dataset = dataset.rename_column("label", "labels")

In [8]:
dataset["train"].set_format("torch")
dataset["test"].set_format("torch")

In [9]:
_train_transform = transforms.Compose([
    transforms.RandomRotation(degrees=20),
    transforms.RandomHorizontalFlip(),
    transforms.RandomResizedCrop(size=(image_size, image_size), scale=(0.6, 1.0), antialias=True),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                         std=[0.229, 0.224, 0.225]),
])

def train_transform(ex):
    ex["pixel_values"] = [_train_transform(image) for image in ex["image"]]
    return ex

_test_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                         std=[0.229, 0.224, 0.225]),
])

def test_transform(ex):
    ex["pixel_values"] = [_test_transform(image) for image in ex["image"]]
    return ex

In [10]:
dataset["train"].set_transform(train_transform)
dataset["test"].set_transform(test_transform)

## Función de pérdida ponderada y metrica

Hugging Face usa AdamW por defecto

In [11]:
weights = torch.tensor([0.11354586666416572, 0.321527676806911, 0.45105396153855953, 0.11387249499036363]).to(DEVICE)
loss_fct = nn.CrossEntropyLoss(weight=weights)

def compute_metrics(pred):
    preds = np.argmax(pred.predictions, axis=1)
    labels = pred.label_ids
    return {
        "accuracy": sklearn.metrics.accuracy_score(labels, preds),
        "f1": sklearn.metrics.f1_score(labels, preds, average="weighted")
    }

## Collate

In [12]:
def collate_fn(examples):
    pixel_values = torch.stack([example["pixel_values"] for example in examples]).to(DEVICE)
    labels = torch.tensor([example["labels"] for example in examples]).to(DEVICE)
    return {"pixel_values": pixel_values, "labels": labels}

## Entrenamiento (Argumentos y )

In [13]:
model_output_temp=f"~/tmp/models/{model_output_name}/"

trainer_args = TrainingArguments(
    model_output_temp,
    eval_strategy = "epoch",
    save_strategy = "epoch",
    logging_strategy="epoch",
    fp16=True,
    per_device_train_batch_size=22,
    per_device_eval_batch_size=64,
    num_train_epochs=25,
    load_best_model_at_end=True,
    remove_unused_columns=False,
    metric_for_best_model="f1",
    greater_is_better=True,
    push_to_hub=False,
    report_to=[],
)

In [14]:
class WeightedLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss = loss_fct(logits.view(-1, model.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

trainer = WeightedLossTrainer(
    model=model,
    args=trainer_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=collate_fn,
    compute_metrics=compute_metrics
)

## Entrenar y guardar modelo

In [ ]:
#trainer.train()
#trainer.save_model(model_output)
#trainer.save_state()